## 1. Import Required Libraries
We start by loading the essential Python libraries:

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

sns.set_style('whitegrid')

## 2. Load and Inspect the Dataset
We read the `aapl_data.csv` file into a pandas DataFrame.

In [ ]:
df = pd.read_csv('aapl_data.csv', sep='\\t')
df.head()

Keep only AAPL rows

In [ ]:

df = df[df['Name'] == 'AAPL'].copy()

## 3. Data Inspection
We check the dataset:
- `info()` tells us info
- `isnull().sum()` counts missing values
- `describe()` gives a summary

In [ ]:
df.info()
df.isnull().sum()
df.describe()

## 4. Date Range
We extract the first and last day to understand the period.

In [ ]:
print("Earliest date:", df['Date'].min())
print("Latest date:", df['Date'].max())

(Removes duplicate date)

In [ ]:
df = df.drop_duplicates(subset='Date', keep='last')
df.sort_values('Date', inplace=True)

## 5. Closing Price Trend
A simple line chart of closing prices over time reveals the big picture:

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df['Date'], df['Close'], linewidth=0.8, color='steelblue')
plt.title('AAPL Closing Price Over Time', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.grid(True, alpha=0.3)
plt.show()

## 6. Daily Returns Distribution
**Daily return** = percentage change from one close to the next.

In [ ]:
# Remove any previous Daily_Return column to avoid conflicts
df['Date'] = pd.to_datetime(df['Date'])

# 2. Calculate daily returns from Adjusted Close
df['Daily_Return'] = df['Adj Close'].pct_change() * 100

# 3. Find the indices of the maximum and minimum return
max_idx = df['Daily_Return'].idxmax()
min_idx = df['Daily_Return'].idxmin()

# 4. Print the results (now safe)
print(f"📈 Best day  : +{df.loc[max_idx, 'Daily_Return']:.2f}% on {df.loc[max_idx, 'Date'].date()}")
print(f"📉 Worst day :  {df.loc[min_idx, 'Daily_Return']:.2f}% on {df.loc[min_idx, 'Date'].date()}")


In [ ]:
# Calculate daily percentage change
df['Daily_Return'] = df['Close'].pct_change() * 100

# Plot histogram of daily returns
plt.figure(figsize=(10, 5))
sns.histplot(df['Daily_Return'].dropna(), bins=150, kde=True, color='green')
plt.title('Distribution of AAPL Daily Returns (%)', fontsize=14)
plt.xlabel('Daily Return (%)')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)
plt.xlim(-13, 13)  
plt.show()

## 7. 30‑Day Rolling Volatility
Volatility = standard deviation of daily returns over a rolling window.

In [ ]:
# 30-day rolling standard deviation of daily returns
df['Rolling_Vol_30'] = df['Daily_Return'].rolling(30).std()

# Plot rolling volatility
plt.figure(figsize=(12, 4))
plt.plot(df['Date'], df['Rolling_Vol_30'], color='orange', linewidth=1)
plt.title('AAPL 30‑Day Rolling Volatility', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Volatility (%)')
plt.grid(True, alpha=0.3)
plt.show()

## Phase 2: Predicting Next‑Day Direction

In [12]:
# Create the prediction target (next-day direction)
# Bring tomorrow's adjusted close into today's row
df['Next_Close'] = df['Adj Close'].shift(-1)

# Target = 1 if tomorrow's close > today's close, else 0
df['Target'] = (df['Next_Close'] > df['Adj Close']).astype(int)

# Check the last few rows to confirm
print("Last 5 rows with Target:")
display(df[['Date', 'Adj Close', 'Next_Close', 'Target']].tail(5))

Last 5 rows with Target:


,Date,Adj Close,Next_Close,Target
2938,2021-09-03,154.300003,156.690002,1
2939,2021-09-07,156.690002,155.110001,0
2940,2021-09-08,155.110001,154.070007,0
2941,2021-09-09,154.070007,148.970001,0
2942,2021-09-10,148.970001,NaN,0


In [13]:
# =============================================================================
# FEATURE ENGINEERING – using only PAST information
# =============================================================================

# 1. Lagged returns (yesterday's return, avg of last 3, avg of last 7)
df['Return_Lag1'] = df['Daily_Return'].shift(1)
df['Return_Lag3'] = df['Daily_Return'].rolling(3).mean().shift(1)
df['Return_Lag7'] = df['Daily_Return'].rolling(7).mean().shift(1)

# 2. Volume change vs previous day (shifted)
df['Volume_Change'] = df['Volume'].pct_change().shift(1)

# 3. Recent volatility (7‑day and 30‑day std, shifted)
df['Volatility_7'] = df['Daily_Return'].rolling(7).std().shift(1)
df['Volatility_30'] = df['Daily_Return'].rolling(30).std().shift(1)

# 4. Price ratios from today's bar (shifted so they're past)
df['High_Low_Ratio'] = (df['High'] / df['Low']).shift(1)
df['Close_Open_Ratio'] = (df['Close'] / df['Open']).shift(1)

# 5. Day of week (known at the start of the day, no shift needed)
df['DayOfWeek'] = df['Date'].dt.dayofweek

# =============================================================================
# CREATE A CLEAN MODEL DATASET (drop rows with NaN from rolling/shifts)
# =============================================================================
features_list = [
    'Return_Lag1', 'Return_Lag3', 'Return_Lag7',
    'Volume_Change', 'Volatility_7', 'Volatility_30',
    'High_Low_Ratio', 'Close_Open_Ratio', 'DayOfWeek'
]

df_model = df.dropna(subset=['Target'] + features_list).copy()

print("Model‑ready rows:", len(df_model))
print("Columns:", df_model.columns.tolist())

Model‑ready rows: 2912
Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Name', 'Daily_Return', 'Return_Lag1', 'Return_Lag3', 'Return_Lag7', 'Volume_Change', 'Volatility_7', 'Volatility_30', 'High_Low_Ratio', 'Close_Open_Ratio', 'DayOfWeek', 'Next_Close', 'Target']


In [14]:
# Chronological split – train on past, test on future
split_index = int(0.8 * len(df_model))   # 80% train, 20% test

train = df_model.iloc[:split_index]
test  = df_model.iloc[split_index:]

print(f"Training: {train['Date'].min().date()} → {train['Date'].max().date()} ({len(train)} rows)")
print(f"Testing : {test['Date'].min().date()} → {test['Date'].max().date()} ({len(test)} rows)")

Training: 2010-02-18 → 2019-05-20 (2329 rows)
Testing : 2019-05-21 → 2021-09-10 (583 rows)


In [15]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

features = features_list   # already defined earlier
X_train, y_train = train[features], train['Target']
X_test,  y_test  = test[features],  test['Target']

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy on unseen test data: {accuracy:.2%}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy on unseen test data: 52.14%

Confusion Matrix:
[[ 44 225]
 [ 54 260]]

Classification Report:
              precision    recall  f1-score   support

           0       0.45      0.16      0.24       269
           1       0.54      0.83      0.65       314

    accuracy                           0.52       583
   macro avg       0.49      0.50      0.45       583
weighted avg       0.50      0.52      0.46       583

